# Rebuild dataset_audit.yxmd in pandas, step by step

1. **Settings**: set `FOLDER` to the workflow's folder, and change any file path to your own copy.
2. Run **Helpers** and **Load**: every file the workflow opens is read once, into a `src_…` table.
3. Then go branch by branch, as in the migration guide. Each code cell starts from the table the branch receives and lists its steps as comments. Under each comment, paste that step's pandas lines from the guide (the folded line under the branch's table), run the cell, and compare `df.head()` with what Alteryx shows for that tool.
4. The last cell saves the outputs and compares one with the file the Alteryx workflow saved.

Step numbers (2.16) and table names (`s2_16`, `src_…`) are the guide's, so a pasted line finds its inputs.

In [ ]:
# Settings: the workflow's folder and the files it opens. Change a path to your own copy.
import os

FOLDER = r"C:\path\to\workflow_folder"   # the workflow's relative paths start here
OUT = os.path.join(FOLDER, "python_out")   # where the last cell saves the outputs

DATASETS_EXPORT_CSV = os.path.join(FOLDER, "Input", "datasets_export.csv")   # Input\datasets_export.csv; opened at step 1.1
OLD_RULES_CSV = os.path.join(FOLDER, "Input", "old_rules.csv")   # Input\old_rules.csv; opened at step 3.1

## Helpers

Alteryx compares text ignoring case, and treats null and the empty string as different; the guide's pandas lines use these two helpers.

In [ ]:
import functools, glob, operator, os, subprocess
import numpy as np
import pandas as pd

pd.options.mode.copy_on_write = True   # df = s2_6, then changing df, leaves s2_6 as it was

def empty(s):
    """Alteryx IsEmpty: null or the empty string."""
    return s.isna() | (s == "")

def eq(a, b):
    """Alteryx "=": text is compared ignoring case; null equals null."""
    fold = lambda x: x.str.casefold() if isinstance(x, pd.Series) and x.dtype == object else (
        x.casefold() if isinstance(x, str) else x)
    null = lambda x: x.isna() if isinstance(x, pd.Series) else x is None
    return (fold(a) == fold(b)) | (null(a) & null(b))

## Load

Every file the workflow opens, read once. In Alteryx each is its own Input tool; the guide shows a file that only feeds a Join, Find Replace or Union inside that step.

In [ ]:
# step 1.1: datasets_export.csv
src_datasets_export = pd.read_csv(DATASETS_EXPORT_CSV, sep=',', encoding="utf-8-sig", dtype=str, keep_default_na=False).replace("", None).astype({"DQScore": float, "SizeGB": float})
# step 2.15 (inside): a typed table
src_typed_owner_team = pd.DataFrame([['ana', 'Finance'], ['bo', 'Risk']], columns=["Owner", "Team"])

print("src_datasets_export", src_datasets_export.shape)
print("src_typed_owner_team", src_typed_owner_team.shape)

## 1. Import and consolidate

- step 1.1 opens `datasets_export.csv`
- step 1.4 saves `Consolidated Datasets.xlsx` › sheet `Datasets`

### Branch 1.1 — Read datasets_export.csv

| Step | Tool | What it does |
|---|---|---|
| 1.1 | Input Data | opens `datasets_export.csv` |
| 1.2 | Data Cleansing | cleans the columns [Name], [Owner], [Tags]: remove leading and trailing spaces |
| 1.3 | Formula | fills column **Load Date** = DateTimeToday() |
| 1.4 | Output Data | saves the table to `Consolidated Datasets.xlsx`, sheet `Datasets` (replaces the sheet) |

In [ ]:
# Branch 1.1 — Read datasets_export.csv
df = src_datasets_export
# 1.1 Input Data — opens `datasets_export.csv`   (already loaded as src_datasets_export)

# 1.2 Data Cleansing — cleans the columns [Name], [Owner], [Tags]: remove leading and trailing spaces   (paste the pandas lines from the guide)

# 1.3 Formula — fills column Load Date = DateTimeToday()   (paste the pandas lines from the guide)

# 1.4 Output Data — saves the table to `Consolidated Datasets.xlsx`, sheet `Datasets` (replaces the sheet)   (paste the pandas lines from the guide)

s1_4 = df
df.head()

## 2. Dataset audit

- step 2.6 opens each file like `template.xlsx` › sheet `Datasets`
- step 2.27 saves `Dataset audit.xlsx` › sheet `Issues`

### Branch 2.1 — Newest consolidated file

| Step | Tool | What it does |
|---|---|---|
| 2.1 | Directory “Newest consolidated file” | lists the files in folder `Output` named `*.xlsx` |
| 2.2 | Filter | keeps only the rows where [FileName] contains "Consolidated" and [FileName] does not contain ".… |
| 2.3 | Sort | sorts by [CreationTime] Z→A |
| 2.4 | Sample | keeps the first row |
| 2.5 | Formula | fills column **Source** = [FullPath] + "\|\|\|Datasets" |
| 2.6 | Dynamic Input | opens every file listed in column [Source] (the column holds the whole path; sheet as in the template `template.xlsx`, sheet `Datasets`), and stacks them |

In [ ]:
# Branch 2.1 — Newest consolidated file
# 2.1 Directory “Newest consolidated file” — lists the files in folder `Output` named `*.xlsx`   (paste the pandas lines from the guide)

# 2.2 Filter — keeps only the rows where [FileName] contains "Consolidated" and [FileName] does not contain ".…   (paste the pandas lines from the guide)

# 2.3 Sort — sorts by [CreationTime] Z→A   (paste the pandas lines from the guide)

# 2.4 Sample — keeps the first row   (paste the pandas lines from the guide)

# 2.5 Formula — fills column Source = [FullPath] + "|||Datasets"   (paste the pandas lines from the guide)

# 2.6 Dynamic Input — opens every file listed in column [Source] (the column holds the whole path; sheet as in the template `template.xlsx`, sheet `Datasets`), and stacks them   (paste the pandas lines from the guide)

s2_6 = df
df.head()

### Branch 2.2 — CreationMode

| Step | Tool | What it does |
|---|---|---|
| 2.7 | Filter “CreationMode” | keeps only the rows where [CreationMode] = "Manual" |
| 2.8 | Formula | fills column **Issue** = "Created by hand" |
| 2.9 | Select | keeps only the columns [Name], [Owner], [Issue] |

In [ ]:
# Branch 2.2 — CreationMode
df = s2_6
# 2.7 Filter “CreationMode” — keeps only the rows where [CreationMode] = "Manual"   (paste the pandas lines from the guide)

# 2.8 Formula — fills column Issue = "Created by hand"   (paste the pandas lines from the guide)

# 2.9 Select — keeps only the columns [Name], [Owner], [Issue]   (paste the pandas lines from the guide)

s2_9 = df
df.head()

### Branch 2.3 — Scenario

| Step | Tool | What it does |
|---|---|---|
| 2.10 | Filter “Scenario” | keeps only the rows where [Scenario] is empty |
| 2.11 | Formula | fills column **Issue** = "No scenario" |
| 2.12 | Summarize | groups the rows by [Owner], [Issue] and calculates **Datasets**, **Name** (like a pivot table) |

In [ ]:
# Branch 2.3 — Scenario
df = s2_6
# 2.10 Filter “Scenario” — keeps only the rows where [Scenario] is empty   (paste the pandas lines from the guide)

# 2.11 Formula — fills column Issue = "No scenario"   (paste the pandas lines from the guide)

# 2.12 Summarize — groups the rows by [Owner], [Issue] and calculates Datasets, Name (like a pivot table)   (paste the pandas lines from the guide)

s2_12 = df
df.head()

### Branch 2.4 — Tags

| Step | Tool | What it does |
|---|---|---|
| 2.13 | Filter “Tags” | keeps only the rows where [Tags] does not contain "certified" |
| 2.14 | Formula | fills column **Issue** (see the rule) |
| 2.15 | Join | matches its rows with a typed table ([Owner], [Team]; 2 rows) on [Owner] |

In [ ]:
# Branch 2.4 — Tags
df = s2_6
# 2.13 Filter “Tags” — keeps only the rows where [Tags] does not contain "certified"   (paste the pandas lines from the guide)

# 2.14 Formula — fills column Issue (see the rule)   (paste the pandas lines from the guide)

# 2.15 Join — matches its rows with a typed table ([Owner], [Team]; 2 rows) on [Owner]   (paste the pandas lines from the guide)

# step 2.15's lines make s2_15_join
s2_15_join.head()

### Branch 2.5 — Data Quality

| Step | Tool | What it does |
|---|---|---|
| 2.16 | Filter “Data Quality” | keeps only the rows where [DQScore] < 0.8 |
| 2.17 | Formula | fills column **Issue** = "Low data quality: " + ToString([DQScore], 2) |
| 2.18 | RegEx | applies the pattern `(\w+)_v\d+` to [Name] (ParseSimple) |

In [ ]:
# Branch 2.5 — Data Quality
df = s2_6
# 2.16 Filter “Data Quality” — keeps only the rows where [DQScore] < 0.8   (paste the pandas lines from the guide)

# 2.17 Formula — fills column Issue = "Low data quality: " + ToString([DQScore], 2)   (paste the pandas lines from the guide)

# 2.18 RegEx — applies the pattern `(\w+)_v\d+` to [Name] (ParseSimple)   (paste the pandas lines from the guide)

s2_18 = df
df.head()

### Branch 2.6 — Virtualizable

| Step | Tool | What it does |
|---|---|---|
| 2.19 | Filter “Virtualizable” | keeps only the rows where [Virtualizable] = "N" and [SizeGB] > 100 |
| 2.20 | Formula | fills column **Issue** = "Large and not virtualizable" |
| 2.21 | Sort | sorts by [SizeGB] Z→A |

In [ ]:
# Branch 2.6 — Virtualizable
df = s2_6
# 2.19 Filter “Virtualizable” — keeps only the rows where [Virtualizable] = "N" and [SizeGB] > 100   (paste the pandas lines from the guide)

# 2.20 Formula — fills column Issue = "Large and not virtualizable"   (paste the pandas lines from the guide)

# 2.21 Sort — sorts by [SizeGB] Z→A   (paste the pandas lines from the guide)

s2_21 = df
df.head()

### Branch 2.7 — Read/Write

| Step | Tool | What it does |
|---|---|---|
| 2.22 | Filter “Read/Write” | keeps only the rows where [Access] is one of "RW", "W" |
| 2.23 | Formula | fills column **Issue** = "Write only" when [Access] = "W", otherwise "Read/write" |
| 2.24 | Select | deletes the columns [Access] |

In [ ]:
# Branch 2.7 — Read/Write
df = s2_6
# 2.22 Filter “Read/Write” — keeps only the rows where [Access] is one of "RW", "W"   (paste the pandas lines from the guide)

# 2.23 Formula — fills column Issue = "Write only" when [Access] = "W", otherwise "Read/write"   (paste the pandas lines from the guide)

# 2.24 Select — deletes the columns [Access]   (paste the pandas lines from the guide)

s2_24 = df
df.head()

### Branch 2.8 — All issues

| Step | Tool | What it does |
|---|---|---|
| 2.25 | Union “All issues” | stacks the tables 2.9, 2.12, 2.15 (J: matched), 2.18, 2.21, 2.24 (columns matched by name) |
| 2.26 | Formula | fills column **Audit Date** = DateTimeToday() |
| 2.27 | Output Data | saves the table to `Dataset audit.xlsx`, sheet `Issues` (replaces the sheet) |

In [ ]:
# Branch 2.8 — All issues
# 2.25 Union “All issues” — stacks the tables 2.9, 2.12, 2.15 (J: matched), 2.18, 2.21, 2.24 (columns matched by name)   (paste the pandas lines from the guide)

# 2.26 Formula — fills column Audit Date = DateTimeToday()   (paste the pandas lines from the guide)

# 2.27 Output Data — saves the table to `Dataset audit.xlsx`, sheet `Issues` (replaces the sheet)   (paste the pandas lines from the guide)

s2_27 = df
df.head()

## Save and compare

In [ ]:
# Save what each Output step saves (uncomment to write the files into OUT):
# os.makedirs(OUT, exist_ok=True)
# s1_4.to_excel(os.path.join(OUT, "Consolidated Datasets.xlsx"), sheet_name="Datasets", index=False)
# s2_27.to_excel(os.path.join(OUT, "Dataset audit.xlsx"), sheet_name="Issues", index=False)

# Compare step 2.27's rows with the file the Alteryx workflow saved (point this at a copy of it):
legacy_file = os.path.join(FOLDER, "Output", "Dataset audit.xlsx")
if os.path.exists(legacy_file):
    legacy = pd.read_excel(legacy_file, sheet_name="Issues")
    print("rows:", len(s2_27), "here,", len(legacy), "in the legacy file; columns only here:", sorted(set(s2_27.columns) - set(legacy.columns)), "only there:", sorted(set(legacy.columns) - set(s2_27.columns)))
    common = [c for c in s2_27.columns if c in legacy.columns]
    print(s2_27[common].reset_index(drop=True).compare(legacy[common]) if len(s2_27) == len(legacy) else "the row counts differ")

## Containers disabled in the saved file

Alteryx skips these until someone enables them (teams often run one container at a time). They are here so each can be rebuilt and run on its own; run their cells only when you want them.

In [ ]:
# step 3.1: old_rules.csv
src_old_rules = pd.read_csv(OLD_RULES_CSV, sep=',', dtype=str, keep_default_na=False).replace("", None)

print("src_old_rules", src_old_rules.shape)

## 3. Old checks (disabled in the saved file)

- step 3.1 opens `old_rules.csv`

### Branch 3.1 — Read old_rules.csv

| Step | Tool | What it does |
|---|---|---|
| 3.1 | Input Data | opens `old_rules.csv` |
| 3.2 | Formula | fills column **Old** = 1 |

In [ ]:
# Branch 3.1 — Read old_rules.csv
df = src_old_rules
# 3.1 Input Data — opens `old_rules.csv`   (already loaded as src_old_rules)

# 3.2 Formula — fills column Old = 1   (paste the pandas lines from the guide)

s3_2 = df
df.head()